In [77]:
from datetime import date, timedelta
import requests
import pandas as pd

El siguiente bloque compilará los siguientes parámetros ENSO:
- ONI
- Niño 1
- Niño 3
- Niño 3.4
- Niño 4

Serán guardados por año y mes

In [88]:
urls = ["https://www.psl.noaa.gov/data/correlation/oni.data",
        "https://www.psl.noaa.gov/data/correlation/nina1.anom.data",
        "https://www.psl.noaa.gov/data/correlation/nina3.anom.data",
        "https://www.psl.noaa.gov/data/correlation/nina34.anom.data",
        "https://www.psl.noaa.gov/data/correlation/nina4.anom.data"]
labels = ["oni", "nino1", "nino3", "nino34", "nino4"]

datasets = []                                 # Conjunto de datasets recopilados de cada parámetro
años = []                                     # Primer y último año de cada parámetro

for url in urls:                              # Revisar cada URL
  text = requests.get(url).text.splitlines()  # Obtener líneas de texto
  years = text[0].split()                     # Obtener rango de años
  year, month, data = [], [], []              # Inicializar las variables a recopilar

  diff = int(years[1]) - int(years[0])
  for y in range(diff + 1):                   # Por cada año
    for m in range(1, 13):                    # y por cada mes
      year.append(int(years[0]) + y)
      month.append(m)
      data.append(text[y+1].split()[m])       # Recopilar un dato puntual

  datasets.append([year, month, data])        # Al terminar añadirlo a datasets
  años.append(int(years[0]))                  # Primer y último año de ese parámetro seleccionado
  años.append(int(years[1]))

year, month = [], []                          # Inicializar el tiempo
data = [[], [], [], [], []]                   # Crear casillas vacías

for y in range(min(años), max(años) + 1):     # Utilizar el rango años más extenso
  for m in range(1, 13):                      # Barrer cada mes
    month.append(m)                           # Rellenar los índices de año y mes
    year.append(y)
    data[0].append(0)                         # Rellenar las casillas vacías con ceros
    data[1].append(0)
    data[2].append(0)
    data[3].append(0)
    data[4].append(0)

for i in range(len(datasets)):                # Bucket sort de cada dataset recopilado
  for j in range(len(datasets[i][0])):
    data[i][datasets[i][1][j] + (datasets[i][0][j]-min(años))*12 - 1] = datasets[i][2][j]

df = pd.DataFrame({                           # Luego, crear y guardar el dataset procesado
    "year" : year, "month" : month,
    "oni" : data[0], "nino1" : data[1],
    "nino3" : data[2], "nino34" : data[3],
    "nino4" : data[4]
})

df.to_csv("enso_indices_por_mes.csv", index=False)

Se definirán las funciones para convertir a semanas epidemiológicas. Nótese que:

- inician en domingo y terminan en sábado
- la primera semana tiene que tener al menos 4 días en enero
- para designar la primera semana se ubica el primer sábado
- antes de ese día teine que tener 3 o más días

Para mayor información visite: https://www.paho.org/sites/default/files/2016-cha-calendario-epidemiologico.pdf


In [89]:
def inicio_semana_1(y):
    enero_4 = date(y, 1, 4)               # La primera semana tiene que tener al menos 4 días de enero
    sabado = enero_4 + timedelta(         # Diferencia entre sábado y día de semana
        days=(5 - enero_4.weekday()) % 7) # Encuentra el primer sábado con más de 4 días
    return sabado - timedelta(days=6)     # Retrocede al inicio de esa semana

def semana_actual(y, m , d):
  n_semana = ((date(y, m, d) - inicio_semana_1(y)).days // 7) + 1
  return n_semana

Se creará una estructura de datos que contenga:
- 53 semanas por año (aunque algunas en realidad tengan 52)
- Para cada semana el número de días que un mes ocupa
- En caso que una semana tenga días de dos meses, guardar los días ocupados por ese mes

In [90]:
year, week, data = [], [], []

for y in range(2000, 2026): # Crear una estructura que contenga 2000-2025 con 53 semanas en todos
  for w in range(1, 54):
    year.append(y)
    week.append(w)
    data.append([[0, 0, 0], [0, 0, 0]])              # [año, mes, conteo]

n = (date(2025, 12, 31) - date(2000, 1, 1)).days + 1 # días desde inicio y fin de muestra

for i in range(n):
  cd = date(2000, 1, 1) + timedelta(days=i)
  y, m, w, d = cd.year, cd.month, cd.weekday(), cd.day
  wn = semana_actual(y, m, d)

  p = wn + 53 * (y - 2000) - 1

  if wn != 0:                 # Si la semana no corresponde al año anterior
    if data[p][0][0] == 0:    # Si la casilla está desocupada
      data[p][0][0] = y       # Colocar año y mes
      data[p][0][1] = m
      data[p][0][2] += 1      # Incrementar cuántos días de la semana le corresponden al mes
    elif data[p][0][1] == m:  # Si la casilla no está desocupada pero tiene el mismo mes
      data[p][0][2] += 1
    else:                     # Si es un mes diferente
      data[p][1][0] = y       # Incrementar días pertenecientes al otro mes
      data[p][1][1] = m
      data[p][1][2] += 1
  else:                       # Si la semana corresponde al año anterior
    if y > 2000:
      if data[p][0][0] != 0:  # Si la semana anterior no está vacía
        data[p][1][0] = y
        data[p][1][1] = m
        data[p][1][2] += 1
      if data[p-1][0][0] != 0:  # Si la semana pasado anterior no está vacía
        data[p-1][1][0] = y
        data[p-1][1][1] = m
        data[p-1][1][2] += 1


Se barrerá el dataset de índices ENSO y se utilizará la estructura de datos anteriormente creados como punteros para acceder a las filas individuales y ponderar valores en caso una semana tenga días de dos meses distintos.

In [93]:
enso = pd.read_csv("enso_indices_por_mes.csv")
# year, month, oni, nino1, nino3, nino34, nino4

relevant = ["oni", "nino1", "nino3", "nino34", "nino4"] # Variables relevantes
to_build = []               # Lista vacía

for parameter in relevant:  # Barrer todas las variables
  temporal = []             # Lista de valores ponderados

  for line in data:
    d1, d2, p1, p2 = 0, 0, 0, 0
    if line[0][0] != 0:     # Números de días que ocupa el primer mes / en caso la fila no esté vacía
      d1 = enso.loc[(enso["year"] == line[0][0]) & (enso["month"] == line[0][1]), parameter].iloc[0]
      p1 = line[0][2]
    if line[1][0] != 0:     # Número de días que ocupe al segundo mes / en caso la fila no esté vacía
      d2 = enso.loc[(enso["year"] == line[1][0]) & (enso["month"] == line[1][1]), parameter].iloc[0]
      p2 = line[1][2]

    a = round((d1*p1 + d2*p2) / 7, 3) # Realizar la ponderación
    temporal.append(a)                # Añadir a la lista de valores ponderados

  to_build.append(temporal) # Añadir la lista a uno más grande

df = {"year": year, "week": week}     # Crear el dataset y guardarlo
for i in range(len(relevant)):
  df[relevant[i]] = to_build[i]
df=pd.DataFrame(df)
df.head()
df.to_csv("enso_indices_por_semana.csv", index=False)